# 05 - Task B: Rare & Severe-Stage ROP Detection
### UROP: Improving Cross-Dataset Generalization and Severe-Stage Detection in ROP
This notebook implements **Task B** focusing on rare severe-stage categories (Stage 4, Stage 5, AP-ROP):
- Evaluates severe ROP class distribution and extreme imbalance in ROP-VL.
- Compares imbalance-aware strategies:
  1. Ordinary Cross-Entropy Loss
  2. Class-Weighted Cross-Entropy
  3. Focal Loss (Lin et al.)
  4. Class-Balanced Sampling (WeightedRandomSampler)
- Evaluates per-class Recall for Stage 4, Stage 5, and A-ROP.


In [1]:
import os, sys
if "COLAB_GPU" in os.environ or os.path.exists("/content"):
    sys.path.append("/content/drive/MyDrive/ROP_UROP")
    sys.path.append("/content/ROP_UROP")
else:
    sys.path.append(r"d:/UROP")

import torch
import pandas as pd
from src.models import get_model
from src.dataset import build_dataloaders, ROPDataset
from src.losses import get_loss_function
from src.training import Trainer, set_seed
from configs import config

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [2]:
# Load Task B dataset (ROP-VL Active Cases)
taskB_path = config.MANIFESTS_DIR / "taskB_manifest.csv"
taskB_df = pd.read_csv(taskB_path)
print(f"Total Task B images: {len(taskB_df)}")
print("Severity distribution:")
print(taskB_df["severity_label"].value_counts())


Total Task B images: 1052
Severity distribution:
severity_label
mild        574
moderate    241
severe      237
Name: count, dtype: int64


In [3]:
# Compare Imbalance Handling Strategies (Multiclass Severity: mild, moderate, severe)
train_loader, val_loader = build_dataloaders(taskB_df, val_fold=0, batch_size=config.BATCH_SIZE, task="severity_3class")

# Calculate class counts for loss weighting
class_counts = {0: sum(taskB_df["severity_label"]=="mild"), 
                1: sum(taskB_df["severity_label"]=="moderate"), 
                2: sum(taskB_df["severity_label"]=="severe")}
print("Class counts:", class_counts)

# Train ResNet-50 with Focal Loss for Severe ROP
model = get_model("resnet50", num_classes=3, pretrained=True)
criterion = get_loss_function("focal", class_counts=class_counts, gamma=2.0, device=device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    experiment_name="taskB_resnet50_focal_fold0",
    task="severity_3class",
    patience=5
)

epochs = 15 if torch.cuda.is_available() else 1
history = trainer.train(train_loader, val_loader, epochs=epochs)


Class counts: {0: 574, 1: 241, 2: 237}


d:\UROP\src\training.py:71: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.use_amp)



--- Starting Experiment: taskB_resnet50_focal_fold0 on cpu ---
Epochs: 1 | Batch size: 16 | Task: severity_3class


d:\UROP\src\training.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp):
d:\UROP\src\training.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp):


Epoch [01/01] Train Loss: 0.4537 | Val Loss: 0.3673 | Val Acc: 0.5806 | AUROC: 0.0000 | F1: 0.5600 | (519.8s)
  --> Saved new best checkpoint (Metric: 0.5600)

Training completed. Best monitored metric: 0.5600


In [4]:
# Evaluate per-class performance (mild, moderate, severe)
trainer.load_best_model()
val_loss, metrics = trainer.evaluate(val_loader)
print("Multiclass Severity Performance:")
print(f"Accuracy: {metrics['accuracy']:.4f} | Macro-F1: {metrics['macro_f1']:.4f}")
for cls_name, vals in metrics["per_class"].items():
    print(f"  {cls_name.upper():<10} -> Precision: {vals['precision']:.3f} | Recall: {vals['recall']:.3f} | F1: {vals['f1_score']:.3f} (N={vals['support']})")


[Loaded] Best weights loaded from D:\UROP\checkpoints\taskB_resnet50_focal_fold0_best.pt


d:\UROP\src\training.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp):


Multiclass Severity Performance:
Accuracy: 0.5806 | Macro-F1: 0.5600
  MILD       -> Precision: 0.844 | Recall: 0.542 | F1: 0.660 (N=120)
  MODERATE   -> Precision: 0.355 | Recall: 0.423 | F1: 0.386 (N=52)
  SEVERE     -> Precision: 0.500 | Recall: 0.867 | F1: 0.634 (N=45)
